In [ ]:
# ============================================
# 1. Import thư viện
# ============================================
import json
import torch
import torch.nn as nn
from torchvision.models import resnet50, ResNet50_Weights
from torchvision import transforms
from PIL import Image, ImageOps
import numpy as np
from google.colab import output
from IPython.display import display, Javascript
from io import BytesIO
import base64

# ============================================
# 2. Load class names
# ============================================
with open("/content/drive/MyDrive/Dataset_Kanji_2/class_names.json", "r", encoding="utf-8") as f:
    labels = json.load(f)
idx2label = {i: l for i, l in enumerate(labels)}
num_classes = len(labels)

# ============================================
# 3. Xây dựng model (giống training)
# ============================================
def build_model(num_classes: int):
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.4),
        nn.Linear(in_features, 512),
        nn.ReLU(inplace=True),
        nn.BatchNorm1d(512),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    return model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = build_model(num_classes).to(device)

# Load checkpoint
state = torch.load("/content/drive/MyDrive/Dataset_Kanji_2/best_model_69 (1).pth", map_location=device)
if any(k.startswith("module.") for k in state.keys()):
    from collections import OrderedDict
    new_state = OrderedDict()
    for k, v in state.items():
        new_state[k.replace("module.", "", 1)] = v
    state = new_state
model.load_state_dict(state)
model.eval()

# ============================================
# 4. Transform ảnh (giống lúc train)
# ============================================
IMG_SIZE = 224
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

# ============================================
# 5. Hàm xử lý ảnh từ canvas
# ============================================
def preprocess_canvas_image(img, target_size=224):
    if img.mode != "RGB":
        img = img.convert("RGB")
    img = img.resize((target_size, target_size))
    return transform(img).unsqueeze(0).to(device)

# ============================================
# 6. Callback: nhận ảnh từ canvas và dự đoán
# ============================================
def save_drawing_callback(data_url):
    try:
        # Decode base64 image
        image_data = base64.b64decode(data_url.split(',')[1])
        img = Image.open(BytesIO(image_data)).convert("L")  # lấy grayscale

        # Không invert nữa (vì canvas đã là chữ trắng nền đen)
        # img = ImageOps.invert(img)

        # Convert về RGB cho đúng transform
        img = img.convert("RGB")

        # Preprocess + predict
        x = preprocess_canvas_image(img)
        with torch.no_grad():
            logits = model(x)
            probs = torch.softmax(logits, dim=1)[0]
            pvals, idxs = torch.topk(probs, 5)

        print("\n===== Kết quả dự đoán từ canvas =====")
        for i, p in zip(idxs, pvals):
            print(f"{idx2label[i.item()]}: {p.item():.4f}")

    except Exception as e:
        print(f"Lỗi: {e}")


# Đăng ký callback
output.register_callback('notebook.save_drawing', save_drawing_callback)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 107MB/s]


In [ ]:
# ============================================
# 7. Hàm tạo canvas trong Colab
# ============================================
def create_drawing_canvas():
    js_code = """
    var canvas = document.createElement('canvas');
    canvas.width = 400;
    canvas.height = 400;
    canvas.style.border = '2px solid black';
    canvas.style.backgroundColor = 'black';
    document.body.appendChild(canvas);
    document.body.appendChild(document.createElement('br'));

    var ctx = canvas.getContext('2d');
    ctx.fillStyle = 'black';
    ctx.fillRect(0, 0, canvas.width, canvas.height);
    ctx.strokeStyle = 'white';   // chữ trắng
    ctx.lineWidth = 20;
    ctx.lineCap = 'round';

    var drawing = false;
    var lastX = 0;
    var lastY = 0;

    canvas.addEventListener('mousedown', function(e) {
        drawing = true;
        lastX = e.offsetX;
        lastY = e.offsetY;
    });

    canvas.addEventListener('mousemove', function(e) {
        if (drawing) {
            ctx.beginPath();
            ctx.moveTo(lastX, lastY);
            ctx.lineTo(e.offsetX, e.offsetY);
            ctx.stroke();
            lastX = e.offsetX;
            lastY = e.offsetY;
        }
    });

    canvas.addEventListener('mouseup', function() {
        drawing = false;
    });
    canvas.addEventListener('mouseleave', function() {
        drawing = false;
    });

    // Nút xóa
    var clearBtn = document.createElement('button');
    clearBtn.innerHTML = 'Xóa';
    clearBtn.style.margin = '5px';
    clearBtn.onclick = function() {
        ctx.fillStyle = 'black';
        ctx.fillRect(0, 0, canvas.width, canvas.height);
    };
    document.body.appendChild(clearBtn);

    // Nút dự đoán
    var predictBtn = document.createElement('button');
    predictBtn.innerHTML = 'Dự đoán';
    predictBtn.style.margin = '5px';
    predictBtn.onclick = function() {
        var dataURL = canvas.toDataURL('image/png');
        google.colab.kernel.invokeFunction(
            'notebook.save_drawing',
            [dataURL],
            {}
        );
    };
    document.body.appendChild(predictBtn);
    """
    display(Javascript(js_code))

# ============================================
# 8. Khởi chạy
# ============================================
print("👉 Vẽ ký tự vào canvas bên dưới, nhấn 'Dự đoán'")
create_drawing_canvas()

👉 Vẽ ký tự vào canvas bên dưới, nhấn 'Dự đoán'


<IPython.core.display.Javascript object>


===== Kết quả dự đoán từ canvas =====
大: 0.7153
木: 0.2327
太: 0.0213
水: 0.0115
才: 0.0096

===== Kết quả dự đoán từ canvas =====
犬: 0.7446
大: 0.1807
才: 0.0656
木: 0.0047
太: 0.0018

===== Kết quả dự đoán từ canvas =====
沸: 0.6740
湯: 0.2478
温: 0.0233
吊: 0.0200
潟: 0.0119

===== Kết quả dự đoán từ canvas =====
訊: 0.8379
詔: 0.0637
訳: 0.0616
唱: 0.0135
設: 0.0127

===== Kết quả dự đoán từ canvas =====
認: 0.9843
訳: 0.0090
訊: 0.0056
詔: 0.0008
誌: 0.0001

===== Kết quả dự đoán từ canvas =====
畷: 0.8641
暇: 0.1302
瞬: 0.0023
腰: 0.0017
曙: 0.0015

===== Kết quả dự đoán từ canvas =====
暇: 0.6019
畷: 0.3463
瞬: 0.0285
腰: 0.0209
曙: 0.0013

===== Kết quả dự đoán từ canvas =====
濫: 0.9999
澗: 0.0000
週: 0.0000
潤: 0.0000
艦: 0.0000

===== Kết quả dự đoán từ canvas =====
協: 0.9999
帖: 0.0000
耐: 0.0000
噺: 0.0000
附: 0.0000

===== Kết quả dự đoán từ canvas =====
褒: 0.9650
庭: 0.0234
裂: 0.0017
堤: 0.0014
控: 0.0010

===== Kết quả dự đoán từ canvas =====
筆: 0.9973
隼: 0.0021
華: 0.0003
畢: 0.0001
建: 0.0001
